In [1]:
from mask import create_masked_phi
from datasets import load_from_disk

In [2]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "microsoft/Phi-3-mini-128k-instruct"
model = AutoModelForCausalLM.from_pretrained( 
            model_name,  
            device_map="auto",  
            torch_dtype=torch.bfloat16,  
            trust_remote_code=True,  
            ) 
tokenizer = AutoTokenizer.from_pretrained(model_name) 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
target_layers = list(range(24,31))

In [4]:
dataset = load_from_disk("./data/preprocessed/general_generated")
data = dataset[0]
data

{'user': 'What are the primary goals and challenges of space tourism as a commercial industry?',
 'assistant': 'Space tourism aims to make space accessible to civilians, facing significant cost, safety, and regulatory challenges.',
 'role': 'maint_kg'}

In [5]:
target_layers

[24, 25, 26, 27, 28, 29, 30]

In [6]:
role_map = {"maint_kg": 0, 
            "maint_lm": 1,
            "code_lm":1,
            "target_kg":2,}

tokenized_data = tokenizer.apply_chat_template([{"role": "user", "content": data["user"]}, {"role": "assistant", "content": data["assistant"]}], tokenize=True, padding="max_length", max_length=4096, truncation=True, return_tensors="pt")[0]
role = torch.tensor(role_map[data["role"]])


In [7]:
model = create_masked_phi(model, target_layers)

In [8]:
model.train();

In [9]:
tokenized_data

tensor([32000, 32000, 32000,  ..., 29889, 32007, 32000])

In [10]:
with torch.no_grad():
    output = model(input_ids=tokenized_data.to(model.device).unsqueeze(0), output_attentions=True)

You are not running the flash-attention implementation, expect numerical differences.
You are not running the flash-attention implementation, expect numerical differences.


In [11]:
attentions = torch.cat(output.attentions).cpu()
attentions.shape

torch.Size([32, 32, 4096, 4096])

In [19]:
output.logits

tensor([[[nan, nan, nan,  ..., nan, nan, nan],
         [nan, nan, nan,  ..., nan, nan, nan],
         [nan, nan, nan,  ..., nan, nan, nan],
         ...,
         [nan, nan, nan,  ..., nan, nan, nan],
         [nan, nan, nan,  ..., nan, nan, nan],
         [nan, nan, nan,  ..., nan, nan, nan]]], device='cuda:0')